# Setting and loading environment variables

In the new Verily workbench, environment variables are necessary for accessing project and workspace bucket paths, as well as the workspace CDR version. The following commands are from the tutorial on adapting a legacy workspace to the Verily workbench: https://workbench.verily.com/workspaces/all-of-us-tutorial-workspace-workspace-migration-rt/resources/e83a3b31-c469-45ad-948f-9bd754eaee46/Setting_Env_Variables.ipynb

In [ ]:
import json
import subprocess
import os

# --- Extract variables directly into os.environ ---
workspace = json.loads(subprocess.run(
    ["wb", "workspace", "describe", "--format=json"],
    capture_output=True, text=True, check=True
).stdout)

os.environ["GOOGLE_CLOUD_PROJECT"] = workspace["googleProjectId"]

resources = json.loads(subprocess.run(
    ["wb", "resource", "list", "--format=json"],
    capture_output=True, text=True, check=True
).stdout)

os.environ["WORKSPACE_CDR"] = ""

# --- Step 3: Extract workspace resources ---
for r in resources:
    
    # 1. BUCKET LOGIC (Execute for ALL resources)
    if r["resourceType"] == "GCS_BUCKET":
        print(f"Found bucket: id={r['id']}, bucketName={r['bucketName']}")
        
        # Check temporary bucket first to avoid substring conflicts
        if "temporary-workspace-bucket" in r["id"]:
            os.environ["WORKSPACE_TEMP_BUCKET"] = f"gs://{r['bucketName']}"
        elif "workspace-bucket" in r["id"]:
            os.environ["WORKSPACE_BUCKET"] = f"gs://{r['bucketName']}"

    # 2. BQ DATASET LOGIC (Only set if CDR is not already set)
    elif r["resourceType"] in ["BQ_DATASET", "BIGQUERY_DATASET"]:
        # Check if the WORKSPACE_CDR is still an empty string (i.e., not set yet)
        if os.environ.get("WORKSPACE_CDR") == "":
            os.environ["WORKSPACE_CDR"] = f"{r['projectId']}.{r['datasetId']}"
            print(f"Successfully set WORKSPACE_CDR to: {os.environ['WORKSPACE_CDR']}")

# --- Print what we got ---
print("\nVariables extracted:")
for var in ["GOOGLE_CLOUD_PROJECT", "WORKSPACE_BUCKET", 
            "WORKSPACE_TEMP_BUCKET", "WORKSPACE_CDR"]:
    value = os.environ.get(var)
    print(f"{var}: {value if value else 'NOT FOUND'}")


# --- Save to .bashrc (no dictionary needed!) ---
bashrc_path = os.path.expanduser("~/.bashrc")

# Check if .bashrc exists, create it if not
if not os.path.exists(bashrc_path):
    print(f"Creating {bashrc_path}...")
    with open(bashrc_path, "w") as f:
        f.write("# Created by Verily setup script\n")

# Now continue with reading/appending
# Simple approach: just append (will create duplicates if run multiple times)
with open(bashrc_path, "a") as f:
    f.write("\n# Verily Workbench variables\n")
    for var in ["GOOGLE_CLOUD_PROJECT", "WORKSPACE_BUCKET", 
                "WORKSPACE_TEMP_BUCKET", "WORKSPACE_CDR"]:
        value = os.environ.get(var)
        if value:
            f.write(f'export {var}="{value}"\n')

print(f"\n✅ Saved to {bashrc_path}")

Set the workspace bucket directly (as seen in the tutorial: https://workbench.verily.com/workspaces/all-of-us-tutorial-workspace-workspace-migration-rt/resources/e83a3b31-c469-45ad-948f-9bd754eaee46/Migrated_Workspaces_for_RT/04_Migrated_Workspaces_for_RT_(Python).ipynb) 

In [ ]:
# set workspace bucket (migrated path, which will be printed above)
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-f6795176" 

In [ ]:
# test load
import os

# In your notebook, just run this:
with open(os.path.expanduser("~/.bashrc"), 'r') as f:
    for line in f:
        if line.strip().startswith('export '):
            parts = line.strip().replace('export ', '').split('=', 1)
            if len(parts) == 2:
                var_name = parts[0].strip()
                var_value = parts[1].strip().strip("'\"")
                
                # SKIP PATH completely!
                if var_name == 'PATH':
                    continue  # Skip this line
                    
                os.environ[var_name] = var_value

# Now use them
print(f"GOOGLE_CLOUD_PROJECT = {os.environ.get('GOOGLE_CLOUD_PROJECT')}")
print(f"WORKSPACE_BUCKET = {os.environ.get('WORKSPACE_BUCKET')}")

In [ ]:
# check that these loaded correctly
!echo $WORKSPACE_CDR
!echo $WORKSPACE_BUCKET